# NASA EONET Natural Events – Exploratory Data Analysis
**Course:** Exploratory Data Analysis | **Instructor:** Ali Hassan Sherazi

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='tab10')
plt.rcParams.update({'figure.facecolor': '#0e1117', 'axes.facecolor': '#1a1d27',
                     'text.color': '#e0e0e0', 'axes.labelcolor': '#e0e0e0',
                     'xtick.color': '#e0e0e0', 'ytick.color': '#e0e0e0'})

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/NASA_EONET_Natural_Events.csv')
print('Shape:', df.shape)
df.head()

## 2. Data Info & Missing Values

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())

## 3. Data Cleaning

In [ ]:
for col in ['start_date', 'end_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[col] = df[col].dt.tz_localize(None)

for col in ['latitude', 'longitude', 'magnitude', 'duration_days']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dropna(subset=['start_date'], inplace=True)
df.reset_index(drop=True, inplace=True)
print('Clean shape:', df.shape)

## 4. Feature Engineering

In [ ]:
df['year']       = df['start_date'].dt.year
df['month']      = df['start_date'].dt.month
df['month_name'] = df['start_date'].dt.strftime('%b')
df['quarter']    = df['start_date'].dt.quarter
df.head()

## 5. Statistical Summary

In [ ]:
df.describe().round(3)

## 6. Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['category'].value_counts().plot.bar(ax=axes[0], color=sns.color_palette('tab10', 10))
axes[0].set_title('Event Category Counts')
axes[0].tick_params(axis='x', rotation=30)

df['category'].value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%',
    colors=sns.color_palette('tab10', 10), startangle=140)
axes[1].set_ylabel('')
axes[1].set_title('Category Distribution')
plt.tight_layout(); plt.show()

## 7. Yearly Trend

In [ ]:
yearly = df.groupby('year')['id'].count()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(yearly.index, yearly.values, marker='o', color='#4fc3f7', linewidth=2)
ax.fill_between(yearly.index, yearly.values, alpha=0.2, color='#4fc3f7')
ax.set_title('Annual Event Count'); ax.set_xlabel('Year'); ax.set_ylabel('Events')
plt.tight_layout(); plt.show()

## 8. Correlation Heatmap

In [ ]:
corr = df.select_dtypes(include=np.number).corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            linewidths=0.5, linecolor='#0e1117')
ax.set_title('Correlation Matrix')
plt.tight_layout(); plt.show()

## 9. Duration Analysis

In [ ]:
dur_data = df.dropna(subset=['duration_days'])
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(dur_data['duration_days'], bins=40, kde=True, ax=axes[0], color='#4fc3f7')
axes[0].set_title('Duration Histogram')
sns.boxplot(data=dur_data, x='category', y='duration_days', ax=axes[1],
            palette='tab10')
axes[1].set_title('Duration by Category')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

## 10. Key Insights

- **Severe Storms** account for the largest share of events.
- Most events last **1–15 days**; droughts and sea ice events are longest.
- Event frequency has **increased since 2010**, reflecting improved monitoring.
- **Earthquakes** cluster at high magnitudes; **wildfires** span large affected areas.
- Geographic distribution shows dense clusters in the **Pacific Ring of Fire** and tropical storm belts.